In [1]:
from __future__ import annotations

In [2]:
import os
import json
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

In [3]:
# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
@dataclass
class CFG:
    EXP_ID: str = "exp_20260427_058_mikhail_cat2_8_repro"
    SEED: int = 42
    N_SPLITS: int = 5

    TARGET: str = "Irrigation_Need"
    ID_COL: str = "id"

    TRAIN_PATH: str = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH: str = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH: str = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

    OUTPUT_DIR: str = f"/kaggle/working/{EXP_ID}"

    TARGET_MAP: dict = None
    INV_TARGET_MAP: dict = None

    # npy paths
    OOF_018: str = NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"
    PRED_018: str = NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"

    OOF_024: str = NPY_PATH + "oof_xgb_digit_orderedte_min_optuna_biased_refined.npy"
    PRED_024: str = NPY_PATH + "pred_xgb_digit_orderedte_min_optuna_biased_refined.npy"

    OOF_025: str = NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"
    PRED_025: str = NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"

    OOF_026: str = NPY_PATH + "oof_realmlp_pair_te_min_proba_biased_refined.npy"
    PRED_026: str = NPY_PATH + "pred_realmlp_pair_te_min_proba_biased_refined.npy"

    OOF_037: str = NPY_PATH + "oof_dao_xgb_pair_te_posthoc_only_proba.npy"
    PRED_037: str = NPY_PATH + "pred_dao_xgb_pair_te_posthoc_only_proba.npy"

    OOF_039: str = NPY_PATH + "oof_xgb_formula_logit_min_proba.npy"
    PRED_039: str = NPY_PATH + "pred_xgb_formula_logit_min_proba.npy"

    OOF_041: str = NPY_PATH + "oof_trompt_strict_min_proba_biased.npy"
    PRED_041: str = NPY_PATH + "pred_trompt_strict_min_proba_biased.npy"

    OOF_048: str = NPY_PATH + "oof_utaazu_lgbm_min_repro_proba_biased.npy"
    PRED_048: str = NPY_PATH + "pred_utaazu_lgbm_min_repro_proba_biased.npy"
    
    OOF_051: str = NPY_PATH + "oof_xgb_log_orderedte_min_046bparams_biased_refined.npy"
    PRED_051: str = NPY_PATH + "pred_xgb_log_orderedte_min_046bparams_biased_refined.npy"

    OOF_052: str = NPY_PATH + "oof_xgb_raw_orderedte_min_046bparams_biased_refined.npy"
    PRED_052: str = NPY_PATH + "pred_xgb_raw_orderedte_min_046bparams_biased_refined.npy"
    
    OOF_053: str = NPY_PATH + "oof_lgb_digit_te_threshold_min_from_family_proba_biased.npy"
    PRED_053: str = NPY_PATH + "pred_lgb_digit_te_threshold_min_from_family_proba_biased.npy"
    
    OOF_054: str = NPY_PATH + "oof_realmlp_pair_te_threshold_min_from_family_proba_biased.npy"
    PRED_054: str = NPY_PATH + "pred_realmlp_pair_te_threshold_min_from_family_proba_biased.npy"
    
    OOF_055: str = NPY_PATH + "oof_cat_pairwise_te_threshold_min_from_family_proba_biased.npy"
    PRED_055: str = NPY_PATH + "pred_cat_pairwise_te_threshold_min_from_family_proba_biased.npy"
    
    OOF_056: str = NPY_PATH + "oof_cat_pairwise_te_threshold_min_from_family_gpu_default_fix_proba_biased.npy"
    PRED_056: str = NPY_PATH + "pred_cat_pairwise_te_threshold_min_from_family_gpu_default_fix_proba_biased.npy"
      
    OOF_057: str = NPY_PATH + "oof_xgb046b_high_interaction_min_biased.npy"
    PRED_057: str = NPY_PATH + "pred_xgb046b_high_interaction_min_biased.npy"
    
    OOF_058: str = NPY_PATH + "oof_mikhail_cat2_8_thresholded_norm.npy"
    PRED_058: str = NPY_PATH + "pred_mikhail_cat2_8_thresholded_norm.npy"
    
    LR_C: float = 1.0
    HC_MAX_SLOTS: int = 25
    NM_RESTARTS: int = 20

    # Ridge two-stage alpha search for BA
    RIDGE_COARSE_ALPHAS: np.ndarray = None
    RIDGE_FINE_WIDTH_LOG10: float = 0.75
    RIDGE_FINE_POINTS: int = 31

    # submit recommendation
    TOP_K_RECOMMENDED: int = 3
    MIN_GAIN_FOR_RECOMMEND: float = 0.0


CFG.TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
CFG.INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}
CFG.RIDGE_COARSE_ALPHAS = np.logspace(-5, 5, 41)

In [4]:
# ---------------------------------------------------------
# Utils
# ---------------------------------------------------------
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def save_json(obj: dict, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_proba(path: str) -> np.ndarray:
    arr = np.load(path)
    assert arr.ndim == 2 and arr.shape[1] == 3, f"Unexpected shape {arr.shape} for {path}"
    return arr.astype(np.float32)


def rankdata_1d(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method="average").to_numpy()


def mean_raw_corr(a: np.ndarray, b: np.ndarray) -> float:
    vals = []
    for c in range(a.shape[1]):
        vals.append(np.corrcoef(a[:, c], b[:, c])[0, 1])
    return float(np.mean(vals))


def mean_rank_corr(a: np.ndarray, b: np.ndarray) -> float:
    vals = []
    for c in range(a.shape[1]):
        ra = rankdata_1d(a[:, c])
        rb = rankdata_1d(b[:, c])
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))


def softmax(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()


def neg_balanced_acc(raw_weights: np.ndarray, oof_stack: np.ndarray, y: np.ndarray) -> float:
    w = softmax(raw_weights)
    blended = np.tensordot(w, oof_stack, axes=([0], [0]))
    preds = np.argmax(blended, axis=1)
    return -balanced_accuracy_score(y, preds)


def multiclass_stack_features(proba_dict: dict[str, np.ndarray], keys: list[str]) -> np.ndarray:
    return np.hstack([proba_dict[k] for k in keys]).astype(np.float32)

ensure_dir(CFG.OUTPUT_DIR)

In [5]:
# ---------------------------------------------------------
# Load
# ---------------------------------------------------------

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
y = train[CFG.TARGET].map(CFG.TARGET_MAP).values

proba_oof = {
    "018": load_proba(CFG.OOF_018),
    "046b": load_proba(CFG.OOF_024),
    "025": load_proba(CFG.OOF_025),
    "026": load_proba(CFG.OOF_026),
    "037": load_proba(CFG.OOF_037),
    "039": load_proba(CFG.OOF_039),
    "041": load_proba(CFG.OOF_041),
    "048": load_proba(CFG.OOF_048),
    "051": load_proba(CFG.OOF_051),
    "052": load_proba(CFG.OOF_052),
    "053": load_proba(CFG.OOF_053),
    "054": load_proba(CFG.OOF_054),
    "056": load_proba(CFG.OOF_056),
    "057": load_proba(CFG.OOF_057),
    "058": load_proba(CFG.OOF_058),
}

proba_pred = {
    "018": load_proba(CFG.PRED_018),
    "046b": load_proba(CFG.PRED_024),
    "025": load_proba(CFG.PRED_025),
    "026": load_proba(CFG.PRED_026),
    "037": load_proba(CFG.PRED_037),
    "039": load_proba(CFG.PRED_039),
    "041": load_proba(CFG.PRED_041),
    "048": load_proba(CFG.PRED_048),
    "051": load_proba(CFG.PRED_051),
    "052": load_proba(CFG.PRED_052),
    "053": load_proba(CFG.PRED_053),
    "054": load_proba(CFG.PRED_054),
    "056": load_proba(CFG.PRED_056),
    "057": load_proba(CFG.PRED_057),
    "058": load_proba(CFG.PRED_058),
}

S0 = ["046b", "018", "025"]
S1 = S0 + ["048"]
S2 = S0 + ["054"]
S3 = S0 + ["056"]
S4 = S0 + ["048", "054", "056"]
FULL = list(proba_oof.keys())

model_names = S3 + ["057"] + ["058"]
oof_stack = np.array([proba_oof[n] for n in model_names], dtype=np.float32)
test_stack = np.array([proba_pred[n] for n in model_names], dtype=np.float32)

print(f"models : {len(model_names)}")

models : 6


In [6]:
# ---------------------------------------------------------
# Greedy hill climb
# ---------------------------------------------------------
def hill_climb(oof_stack: np.ndarray, y: np.ndarray, model_names: list[str], max_slots: int = 25):
    n_models = len(model_names)
    ensemble_sum = np.zeros_like(oof_stack[0])
    selected = []
    scores_history = []

    for slot in range(max_slots):
        best_score = -1.0
        best_idx = -1

        for i in range(n_models):
            cand_sum = ensemble_sum + oof_stack[i]
            cand_avg = cand_sum / (len(selected) + 1)
            sc = balanced_accuracy_score(y, np.argmax(cand_avg, axis=1))
            if sc > best_score:
                best_score = sc
                best_idx = i

        ensemble_sum = ensemble_sum + oof_stack[best_idx]
        selected.append(best_idx)
        scores_history.append(best_score)
        print(f"  slot {slot+1:02d}: +{model_names[best_idx]} score={best_score:.6f}")

    best_slot = int(np.argmax(scores_history))
    best_score = float(scores_history[best_slot])
    final_selected = selected[: best_slot + 1]

    counts = Counter(final_selected)
    weights = np.zeros(n_models, dtype=np.float32)
    for idx, cnt in counts.items():
        weights[idx] = cnt / len(final_selected)

    return weights, best_score, int(best_slot + 1), scores_history


# ---------------------------------------------------------
# Ridge two-stage search optimized for BA
# ---------------------------------------------------------
def _fit_ridge_multiclass(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_va: np.ndarray,
    X_te: np.ndarray,
    alpha_by_class: list[float],
) -> tuple[np.ndarray, np.ndarray]:
    va_pred = np.zeros((len(X_va), 3), dtype=np.float32)
    te_pred = np.zeros((len(X_te), 3), dtype=np.float32)
    for cls in range(3):
        y_bin = (y_tr == cls).astype(np.float32)
        model = Ridge(alpha=float(alpha_by_class[cls]))
        model.fit(X_tr, y_bin)
        va_pred[:, cls] = model.predict(X_va)
        te_pred[:, cls] = model.predict(X_te)
    return va_pred, te_pred

def _evaluate_alpha_grid_classwise(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_va: np.ndarray,
    y_va: np.ndarray,
    alpha_grid: np.ndarray,
) -> tuple[list[float], list[dict]]:
    best_alphas = []
    alpha_logs = []

    for cls in range(3):
        y_bin = (y_tr == cls).astype(np.float32)
        best_alpha = None
        best_mse = None
        rows = []

        for alpha in alpha_grid:
            model = Ridge(alpha=float(alpha))
            model.fit(X_tr, y_bin)
            pred = model.predict(X_va)
            mse = float(np.mean((pred - (y_va == cls).astype(np.float32)) ** 2))
            rows.append({"alpha": float(alpha), "mse": mse})
            if best_mse is None or mse < best_mse:
                best_mse = mse
                best_alpha = float(alpha)

        best_alphas.append(best_alpha)
        alpha_logs.append({
            "class": cls,
            "best_alpha": best_alpha,
            "best_mse": best_mse,
            "grid_size": int(len(alpha_grid)),
            "top5": sorted(rows, key=lambda x: x["mse"])[:5],
        })

    return best_alphas, alpha_logs

def fit_ridge_oof_two_stage_ba_refine(
    X: np.ndarray,
    y: np.ndarray,
    X_test: np.ndarray,
    n_splits: int = 5,
    seed: int = 42,
    coarse_alphas: np.ndarray | None = None,
    fine_width_log10: float = 0.75,
    fine_points: int = 31,
):
    if coarse_alphas is None:
        coarse_alphas = np.logspace(-5, 5, 41)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_pred = np.zeros((len(X), 3), dtype=np.float32)
    test_pred = np.zeros((len(X_test), 3), dtype=np.float32)
    fold_scores = []
    alpha_info = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        # stage 1: coarse search, classwise based on regression fitness
        coarse_best, coarse_logs = _evaluate_alpha_grid_classwise(
            X_tr=X_tr,
            y_tr=y_tr,
            X_va=X_va,
            y_va=y_va,
            alpha_grid=coarse_alphas,
        )

        # stage 2: fine search around coarse best, classwise
        fine_best = []
        fine_logs = []
        for cls in range(3):
            center = coarse_best[cls]
            lo = np.log10(center) - fine_width_log10
            hi = np.log10(center) + fine_width_log10
            fine_grid = np.logspace(lo, hi, fine_points)

            best_alpha = None
            best_mse = None
            rows = []
            y_bin = (y_tr == cls).astype(np.float32)
            for alpha in fine_grid:
                model = Ridge(alpha=float(alpha))
                model.fit(X_tr, y_bin)
                pred = model.predict(X_va)
                mse = float(np.mean((pred - (y_va == cls).astype(np.float32)) ** 2))
                rows.append({"alpha": float(alpha), "mse": mse})
                if best_mse is None or mse < best_mse:
                    best_mse = mse
                    best_alpha = float(alpha)

            fine_best.append(best_alpha)
            fine_logs.append({
                "class": cls,
                "center_alpha": float(center),
                "best_alpha": best_alpha,
                "best_mse": best_mse,
                "grid_size": int(len(fine_grid)),
                "top5": sorted(rows, key=lambda x: x["mse"])[:5],
            })

        va_pred, te_pred = _fit_ridge_multiclass(
            X_tr=X_tr,
            y_tr=y_tr,
            X_va=X_va,
            X_te=X_test,
            alpha_by_class=fine_best,
        )

        oof_pred[va_idx] = va_pred
        test_pred += te_pred / n_splits

        score = balanced_accuracy_score(y_va, np.argmax(va_pred, axis=1))
        fold_scores.append(float(score))
        alpha_info.append({
            "fold": fold,
            "coarse_best": {f"class_{i}": float(coarse_best[i]) for i in range(3)},
            "fine_best": {f"class_{i}": float(fine_best[i]) for i in range(3)},
            "fold_ba": float(score),
            "coarse_logs": coarse_logs,
            "fine_logs": fine_logs,
        })
        print(
            f"  fold {fold} ridge-2stage BA: {score:.6f} | "
            f"alphas={','.join(f'{a:.6g}' for a in fine_best)}"
        )

    return oof_pred, test_pred, fold_scores, alpha_info


# ---------------------------------------------------------
# LogisticRegression stacking
# ---------------------------------------------------------
def fit_lr_stack_oof(X: np.ndarray, y: np.ndarray, X_test: np.ndarray, c: float = 1.0, n_splits: int = 5, seed: int = 42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_pred = np.zeros((len(X), 3), dtype=np.float32)
    test_pred = np.zeros((len(X_test), 3), dtype=np.float32)
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = LogisticRegression(
            max_iter=2000,
            C=c,
            solver="lbfgs",
            class_weight="balanced",
            random_state=seed,
        )
        model.fit(X_tr, y_tr)

        oof_pred[va_idx] = model.predict_proba(X_va)
        test_pred += model.predict_proba(X_test) / n_splits

        score = balanced_accuracy_score(y_va, np.argmax(oof_pred[va_idx], axis=1))
        fold_scores.append(float(score))
        print(f"  fold {fold} lr-stack BA: {score:.6f}")

    return oof_pred, test_pred, fold_scores

# ---------------------------------------------------------
# Recommendation logic
# ---------------------------------------------------------
def recommend_strategies(results_df: pd.DataFrame, top_k: int = 3, min_gain: float = 0.0) -> pd.DataFrame:
    out = results_df.copy()
    out["recommended_submit"] = False
    out["reason"] = "reference"

    ranked = out.sort_values("cv_score", ascending=False).reset_index(drop=True)
    baseline = float(ranked["cv_score"].max())

    for i in range(min(top_k, len(ranked))):
        idx = ranked.index[i]
        gain = float(ranked.loc[i, "cv_score"] - baseline)
        if gain >= -1e-12 - min_gain:
            pass

    # top-k by CV are recommended by default
    for i in range(min(top_k, len(ranked))):
        ranked.loc[i, "recommended_submit"] = True
        if i == 0:
            ranked.loc[i, "reason"] = "best_cv_among_current_strategies"
        elif ranked.loc[i, "strategy"] == "equal_avg":
            ranked.loc[i, "reason"] = "baseline_diagnostic_and_submit_candidate"
        elif ranked.loc[i, "strategy"].startswith("ridge"):
            ranked.loc[i, "reason"] = "linear_meta_candidate_after_two_stage_alpha_search"
        else:
            ranked.loc[i, "reason"] = "top_k_cv_submit_candidate"

    # non-recommended explicit reasons
    for i in range(len(ranked)):
        if not bool(ranked.loc[i, "recommended_submit"]):
            s = ranked.loc[i, "strategy"]
            if s == "equal_avg":
                ranked.loc[i, "reason"] = "diagnostic_baseline_only"
            elif s.startswith("ridge"):
                ranked.loc[i, "reason"] = "kept_for_reference_but_not_top_k"
            else:
                ranked.loc[i, "reason"] = "cv_below_current_submit_cut"

    return ranked

## Correlation summary

In [7]:
# ---------------------------------------------------------
# Correlation summary
# ---------------------------------------------------------
corr_rows = []
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a = model_names[i]
        b = model_names[j]
        corr_rows.append({
            "a": a,
            "b": b,
            "raw_corr_mean": mean_raw_corr(proba_oof[a], proba_oof[b]),
            "rank_corr_mean": mean_rank_corr(proba_oof[a], proba_oof[b]),
        })

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(f"{CFG.OUTPUT_DIR}/pairwise_corr_summary.csv", index=False)

## Single baselines

In [8]:
# ---------------------------------------------------------
# Single baselines
# ---------------------------------------------------------
single_rows = []
for name in model_names:
    sc = balanced_accuracy_score(y, np.argmax(proba_oof[name], axis=1))
    single_rows.append({"strategy": f"single_{name}", "cv_score": float(sc)})

single_df = pd.DataFrame(single_rows).sort_values("cv_score", ascending=False)
single_df.to_csv(f"{CFG.OUTPUT_DIR}/single_model_scores.csv", index=False)

## Equal average

In [9]:
# ---------------------------------------------------------
# Equal average
# ---------------------------------------------------------
avg_oof = oof_stack.mean(axis=0)
avg_test = test_stack.mean(axis=0)
avg_score = balanced_accuracy_score(y, np.argmax(avg_oof, axis=1))
print(f"\nequal average : {avg_score:.6f}")


equal average : 0.980407


## Nelder-Mead

In [10]:
# ---------------------------------------------------------
# Nelder-Mead
# ---------------------------------------------------------
best_result = None
best_obj = None
for restart in range(CFG.NM_RESTARTS):
    rng = np.random.RandomState(CFG.SEED + restart)
    x0 = rng.randn(len(model_names)) * 0.5
    result = minimize(
        neg_balanced_acc,
        x0,
        args=(oof_stack, y),
        method="Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-7, "fatol": 1e-7},
    )
    if best_result is None or result.fun < best_obj:
        best_result = result
        best_obj = result.fun

nm_weights = softmax(best_result.x)
nm_oof = np.tensordot(nm_weights, oof_stack, axes=([0], [0]))
nm_test = np.tensordot(nm_weights, test_stack, axes=([0], [0]))
nm_score = balanced_accuracy_score(y, np.argmax(nm_oof, axis=1))

nelder_mead_weights = pd.DataFrame({
    "model": model_names, "weight": nm_weights
}).sort_values("weight", ascending=False)

nelder_mead_weights.to_csv(
    f"{CFG.OUTPUT_DIR}/nelder_mead_weights.csv", index=False
)
display(nelder_mead_weights)
print(f"\nnelder-mead : {nm_score:.6f}")

,model,weight
3,056,0.237949
4,057,0.236960
1,018,0.208995
5,058,0.176566
2,025,0.070675
0,046b,0.068854



nelder-mead : 0.980513


## Greedy hill climb

In [11]:
# ---------------------------------------------------------
# Greedy hill climb
# ---------------------------------------------------------
hc_weights, hc_best_score, hc_n_slots, hc_history = hill_climb(
    oof_stack=oof_stack,
    y=y,
    model_names=model_names,
    max_slots=CFG.HC_MAX_SLOTS,
)

hc_oof = np.tensordot(hc_weights, oof_stack, axes=([0], [0]))
hc_test = np.tensordot(hc_weights, test_stack, axes=([0], [0]))
hc_score = balanced_accuracy_score(y, np.argmax(hc_oof, axis=1))

hill_climb_weights = pd.DataFrame({"model": model_names, "weight": hc_weights}).sort_values("weight", ascending=False)
hill_climb_weights.to_csv(
    f"{CFG.OUTPUT_DIR}/hill_climb_weights.csv", index=False
)
display(hill_climb_weights)
print(f"\nhill climb : {hc_score:.6f}")

pd.DataFrame({"slot": list(range(1, len(hc_history) + 1)), "score": hc_history}).to_csv(
    f"{CFG.OUTPUT_DIR}/hill_climb_history.csv", index=False
)
HC_WEIGHT_THRESHOLD = 0.001
selected_model_names = [m for m, w in zip(model_names, hc_weights) if w >= HC_WEIGHT_THRESHOLD]
selected_model_names

  slot 01: +046b score=0.979871
  slot 02: +056 score=0.980393
  slot 03: +058 score=0.980373
  slot 04: +018 score=0.980446
  slot 05: +046b score=0.980485
  slot 06: +056 score=0.980533
  slot 07: +058 score=0.980503
  slot 08: +018 score=0.980446
  slot 09: +057 score=0.980553
  slot 10: +056 score=0.980458
  slot 11: +046b score=0.980554
  slot 12: +046b score=0.980512
  slot 13: +056 score=0.980508
  slot 14: +058 score=0.980505
  slot 15: +056 score=0.980540
  slot 16: +046b score=0.980513
  slot 17: +018 score=0.980531
  slot 18: +057 score=0.980572
  slot 19: +056 score=0.980576
  slot 20: +058 score=0.980570
  slot 21: +056 score=0.980546
  slot 22: +057 score=0.980578
  slot 23: +046b score=0.980526
  slot 24: +018 score=0.980558
  slot 25: +058 score=0.980556


,model,weight
3,056,0.318182
0,046b,0.227273
5,058,0.181818
1,018,0.136364
4,057,0.136364
2,025,0.000000



hill climb : 0.980578


['046b', '018', '056', '057', '058']

In [12]:
pruned_proba_oof = {k: proba_oof[k] for k in selected_model_names}
pruned_proba_pred = {k: proba_pred[k] for k in selected_model_names}

stack_feats_oof_pruned = multiclass_stack_features(pruned_proba_oof, selected_model_names)
stack_feats_test_pruned = multiclass_stack_features(pruned_proba_pred, selected_model_names)

## Ridge blend

In [13]:
# ---------------------------------------------------------
# Ridge blend with two-stage alpha search
# ---------------------------------------------------------
ridge_oof, ridge_test, ridge_fold_scores, ridge_alpha_info = fit_ridge_oof_two_stage_ba_refine(
    X=stack_feats_oof_pruned,
    y=y,
    X_test=stack_feats_test_pruned,
    n_splits=CFG.N_SPLITS,
    seed=CFG.SEED,
    coarse_alphas=CFG.RIDGE_COARSE_ALPHAS,
    fine_width_log10=CFG.RIDGE_FINE_WIDTH_LOG10,
    fine_points=CFG.RIDGE_FINE_POINTS,
)
ridge_score = balanced_accuracy_score(y, np.argmax(ridge_oof, axis=1))
print(f"\nridge 2stage : {ridge_score:.6f}")
save_json({"ridge_alpha_info": ridge_alpha_info}, f"{CFG.OUTPUT_DIR}/ridge_alpha_summary.json")

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.11545e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.11545e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.11545e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.37975e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.11545e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

  fold 1 ridge-2stage BA: 0.978691 | alphas=0.141254,0.141254,316.228


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.53883e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.53883e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.53883e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.76611e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.53883e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

  fold 2 ridge-2stage BA: 0.979689 | alphas=0.141254,0.141254,1.77828e-05


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.0312e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.0312e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.0312e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.0312e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.14932e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.1

  fold 3 ridge-2stage BA: 0.981222 | alphas=0.158489,0.158489,2.81838e-06


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.90269e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.90269e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.90269e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


  fold 4 ridge-2stage BA: 0.979898 | alphas=3.16228e-06,1.99526e-06,2.23872e-06


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.2178e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.2178e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.2178e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.31728e-09): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.00174e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.

  fold 5 ridge-2stage BA: 0.980442 | alphas=1.77828e-06,5.62341e-06,0.0251189

ridge 2stage : 0.979988


## LogisticRegression stacking

In [14]:
# ---------------------------------------------------------
# LogisticRegression stacking
# ---------------------------------------------------------
lr_oof, lr_test, lr_fold_scores = fit_lr_stack_oof(
    X=stack_feats_oof_pruned,
    y=y,
    X_test=stack_feats_test_pruned,
    c=CFG.LR_C,
    n_splits=CFG.N_SPLITS,
    seed=CFG.SEED,
)
lr_score = balanced_accuracy_score(y, np.argmax(lr_oof, axis=1))
print(f"\nstack_lr : {lr_score:.6f}")

  fold 1 lr-stack BA: 0.978770
  fold 2 lr-stack BA: 0.979459
  fold 3 lr-stack BA: 0.981007
  fold 4 lr-stack BA: 0.980246
  fold 5 lr-stack BA: 0.980656

stack_lr : 0.980028


## Summary

In [15]:
# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------
results = [
    {"strategy": "single_046b", "cv_score": balanced_accuracy_score(y, np.argmax(proba_oof["046b"], axis=1))},
    {"strategy": "equal_avg", "cv_score": avg_score},
    {"strategy": "nelder_mead", "cv_score": nm_score},
    {"strategy": "hill_climb", "cv_score": hc_score},
    {"strategy": "ridge_two_stage", "cv_score": ridge_score},
    {"strategy": "stack_lr", "cv_score": lr_score},
]

results_df = pd.DataFrame(results).sort_values("cv_score", ascending=False).reset_index(drop=True)
best_single_046b = float(results_df.loc[results_df["strategy"] == "single_046b", "cv_score"].iloc[0])
results_df["gain_vs_single_046b"] = results_df["cv_score"] - best_single_046b
results_df = recommend_strategies(
    results_df,
    top_k=CFG.TOP_K_RECOMMENDED,
    min_gain=CFG.MIN_GAIN_FOR_RECOMMEND,
)
results_df.to_csv(f"{CFG.OUTPUT_DIR}/blend_strategy_results.csv", index=False)

## Save OOF/Test/Submission per strategy

In [16]:
# ---------------------------------------------------------
# Save OOF / pred / submission per strategy
# ---------------------------------------------------------
strategy_oof = {
    "equal_avg": avg_oof,
    "nelder_mead": nm_oof,
    "hill_climb": hc_oof,
    "ridge_two_stage": ridge_oof,
    "stack_lr": lr_oof,
}
strategy_test = {
    "equal_avg": avg_test,
    "nelder_mead": nm_test,
    "hill_climb": hc_test,
    "ridge_two_stage": ridge_test,
    "stack_lr": lr_test,
}

for name, arr in strategy_oof.items():
    np.save(f"{CFG.OUTPUT_DIR}/oof_{name}.npy", arr)

for name, arr in strategy_test.items():
    np.save(f"{CFG.OUTPUT_DIR}/pred_{name}.npy", arr)
    sub = pd.DataFrame({
        CFG.ID_COL: test[CFG.ID_COL],
        CFG.TARGET: pd.Series(np.argmax(arr, axis=1)).map(CFG.INV_TARGET_MAP),
    })
    sub.to_csv(f"{CFG.OUTPUT_DIR}/submission_{name}.csv", index=False)

## Final Summary

In [17]:
# ---------------------------------------------------------
# Final summary json with recommendation flags
# ---------------------------------------------------------
summary = {
    "exp_id": CFG.EXP_ID,
    "seed": CFG.SEED,
    "n_splits": CFG.N_SPLITS,

    # full stack information
    "full_model_names": model_names,
    "strategies_ranked": results_df.to_dict(orient="records"),

    # full-stack strategies
    "nm_weights_full": dict(zip(model_names, nm_weights.tolist())),
    "hc_weights_full": dict(zip(model_names, hc_weights.tolist())),

    # HC-pruned subset information
    "hc_pruning": {
        "enabled_for_lr_ridge": True,
        "rule": "weight >= threshold",   # 閾値版なら "weight >= threshold"
        "threshold": float(HC_WEIGHT_THRESHOLD),     # 閾値を使うなら追加
        "selected_model_names": selected_model_names,
        "selected_model_count": int(len(selected_model_names)),
    },

    # downstream strategies after HC pruning
    "lr_input_models": selected_model_names,
    "ridge_input_models": selected_model_names,
    "lr_fold_scores": lr_fold_scores,
    "ridge_fold_scores": ridge_fold_scores,
    "lr_input_feature_dim": int(stack_feats_oof_pruned.shape[1]),
    "ridge_input_feature_dim": int(stack_feats_oof_pruned.shape[1]),

    "ridge_search": {
        "coarse_alphas": [float(x) for x in CFG.RIDGE_COARSE_ALPHAS],
        "fine_width_log10": float(CFG.RIDGE_FINE_WIDTH_LOG10),
        "fine_points": int(CFG.RIDGE_FINE_POINTS),
    },

    "notes": {
        "safe_mode": True,
        "all_strategy_submissions_created": True,
        "submit_only_recommended": True,
        "recommendation_rule": f"top_{CFG.TOP_K_RECOMMENDED}_by_cv_with_reason",
        "avg_nm_hc_use_full_stack": True,
        "lr_ridge_use_hc_pruned_stack": True,
    },
}

save_json(summary, f"{CFG.OUTPUT_DIR}/cv_result.json")

print(results_df)
print("\nSaved to:", CFG.OUTPUT_DIR)

          strategy  cv_score  gain_vs_single_046b  recommended_submit  \
0       hill_climb  0.980578             0.000707                True   
1      nelder_mead  0.980513             0.000642                True   
2        equal_avg  0.980407             0.000536                True   
3         stack_lr  0.980028             0.000157               False   
4  ridge_two_stage  0.979988             0.000118               False   
5      single_046b  0.979871             0.000000               False   

                                     reason  
0          best_cv_among_current_strategies  
1                 top_k_cv_submit_candidate  
2  baseline_diagnostic_and_submit_candidate  
3               cv_below_current_submit_cut  
4          kept_for_reference_but_not_top_k  
5               cv_below_current_submit_cut  

Saved to: /kaggle/working/exp_20260427_058_mikhail_cat2_8_repro
